# Multi-Asset CTA Strategy V2 — Transition Strategy

## 02 — Conventional Trend Benchmark and Transition Events

This notebook builds the conventional trend benchmark and the first ex-post transition-event dataset for **Multi-Asset CTA Strategy V2 — Transition Strategy**.

It uses the V2.01 signal-price panel but performs **all rolling calculations on each market's own valid trading calendar**.

This is essential because V2.01 stores markets in a wide panel using the union of international trading dates. Rolling directly across that global calendar can introduce local-holiday NaNs and invalidate moving-average and breakout windows for non-FX markets.

### Research role

The notebook establishes:

1. Slow conventional trend benchmarks:
   - TSMOM
   - Dual moving average
   - Breakout/channel midpoint
   - Majority-vote ensemble

2. Faster versions of the same benchmark families as a falsification control.

3. Confirmed slow-trend reversals.

4. Fast counter-trend candidates occurring while the slow trend remains established.

5. Ex-post labels:
   - `genuine`
   - `failed`

6. Lead-time and forward-return diagnostics.

The key falsification question is:

> Can later V2 transition features distinguish genuine reversals from failed fast counter-trend signals better than simply using faster trend following?

SuperbCommand is deliberately excluded from this notebook.

### Execution requirement

Google Drive is mounted explicitly near the start of this notebook so V2.02 can be run independently in a fresh Colab runtime. Book 01 does not need to be rerun provided its saved V2.01 outputs remain in the project folder.

### Research-window policy

**Primary research window: 2000-01-01 to 2025-12-31.**

Earlier observations are retained only as **warm-up history** so that rolling indicators, trend persistence, and incumbent-state estimates can be properly initialized at the start of the formal research sample.

A market enters formal analysis at the later of:

- its applicable asset-class research start date, or
- the date on which its required indicator warm-up has been completed.

Historical observations before the applicable formal research start may therefore be used for warm-up but cannot generate formal research events.

For traditional asset classes, the formal research start is `2000-01-01`.

Markets that begin after 2000 are retained rather than discarded and enter the analysis once sufficient local-calendar history has accumulated to satisfy the required warm-up.

### Digital-asset extension

V2.02 permits a separate `DIGITAL_ASSETS` asset class. The initial digital-asset universe contains **BTC-USD only**, with a formal digital-asset research start of **2017-01-01**. BTC is never grouped with FX.

Results retain the asset-class label so the full universe can be compared with traditional assets excluding `DIGITAL_ASSETS`.

### Digital-asset entry policy

`DIGITAL_ASSETS` are admitted to the formal V2 research universe only from **2017-01-01** onward.

Historical BTC-USD observations **before 2017-01-01 may be retained and used solely for indicator warm-up**. This allows the conventional trend benchmarks and incumbent-trend state to be fully initialized before digital assets formally enter the research universe.

With the present parameters, the full warm-up requirement is **363 valid local observations**:

`max(252, 300, 252) + 63 = 363`

A digital asset therefore becomes eligible for formal event analysis on the later of:

- `2017-01-01`, or
- the date on which the full 363-observation warm-up requirement has been satisfied.

For BTC-USD, if sufficient pre-2017 history is available, the warm-up may be completed before January 2017. In that case, BTC becomes eligible from the **first valid observation on or after 2017-01-01**, allowing the 2017 cycle, including the December 2017 peak and subsequent transition, to be included in the formal research sample.

The exact eligibility date is calculated from the available local-calendar history rather than hard-coded.

All aggregate outputs retain a separate `DIGITAL_ASSETS` category and can therefore be evaluated both with and without Bitcoin.

In [ ]:
print("RUNNING: V2.02 MARKET-LOCAL-CALENDAR BUILD")

from pathlib import Path
import json
import numpy as np
import pandas as pd

RUNNING: V2.02 MARKET-LOCAL-CALENDAR BUILD


In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted successfully.")
except ImportError:
    print(
        "google.colab is unavailable in this environment. "
        "This notebook is intended to be run in Google Colab."
    )

Mounted at /content/drive
Google Drive mounted successfully.


## 1. Paths and configuration

All outputs are written to:

`/content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.02/`

The parameter set below is intentionally transparent and **not optimised**.

In [ ]:
PROJECT_PARENT = Path(
    "/content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2"
)

V202_ROOT = PROJECT_PARENT / "v2.02"

DATA_DIR = V202_ROOT / "data"
RESULTS_DIR = V202_ROOT / "results"
FIGURES_DIR = V202_ROOT / "figures"
MANIFESTS_DIR = V202_ROOT / "manifests"
CONFIG_DIR = V202_ROOT / "config"

for p in [DATA_DIR, RESULTS_DIR, FIGURES_DIR, MANIFESTS_DIR, CONFIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "research_start": "2000-01-01",
    "digital_assets_universe_start": "2017-01-01",
    "research_end": "2025-12-31",
    "tsmom_slow_days": 252,
    "ma_fast_slow_days": 100,
    "ma_slow_slow_days": 300,
    "breakout_slow_days": 252,
    "tsmom_fast_days": 63,
    "ma_fast_fast_days": 20,
    "ma_slow_fast_days": 100,
    "breakout_fast_days": 63,
    "confirmation_persistence_days": 21,
    "transition_horizon_days": 126,
    "candidate_cooldown_days": 21,
    "established_trend_days": 63,
    "ensemble_min_votes": 2,
}

with open(CONFIG_DIR / "v2_02_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

def find_project_file(filename, preferred_relpaths=()):
    # 1) Try preferred exact locations first.
    for rel in preferred_relpaths:
        candidate = PROJECT_PARENT / rel
        if candidate.exists():
            return candidate

    # 2) Fall back to recursive discovery anywhere under the project parent.
    matches = sorted(PROJECT_PARENT.rglob(filename))

    # Exclude v2.02 or later outputs if filenames happen to collide.
    matches = [
        p for p in matches
        if "v2.02" not in str(p.parent).lower()
    ]

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        print(f"Multiple copies of {filename} found:")
        for p in matches:
            print(" -", p)
        print("Using the first match:", matches[0])
        return matches[0]

    return None

SIGNAL_PATH = find_project_file(
    "v2_01_signal_prices.parquet",
    preferred_relpaths=[
        "v2.01/data/processed/v2_01_signal_prices.parquet",
        "V2.01/data/processed/v2_01_signal_prices.parquet",
    ],
)

UNIVERSE_PATH = find_project_file(
    "v2_master_universe.csv",
    preferred_relpaths=[
        "v2.01/manifests/v2_master_universe.csv",
        "V2.01/manifests/v2_master_universe.csv",
    ],
)

print("V2.02 root:", V202_ROOT)
print("Discovered signal input:", SIGNAL_PATH)
print("Discovered universe manifest:", UNIVERSE_PATH)

if SIGNAL_PATH is None:
    raise FileNotFoundError(
        "Could not find v2_01_signal_prices.parquet anywhere under the project folder. "
        "Confirm Google Drive is mounted and that Book 01 outputs exist, then rerun Book 02."
    )

if UNIVERSE_PATH is None:
    raise FileNotFoundError(
        "Could not find v2_master_universe.csv anywhere under the project folder. "
        "Confirm Google Drive is mounted and that Book 01 outputs exist, then rerun Book 02."
    )

V2.02 root: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.02
Discovered signal input: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.01/data/processed/v2_01_signal_prices.parquet
Discovered universe manifest: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.01/manifests/v2_master_universe.csv


## 2. Load V2.01 signal panel

**Important:** the wide panel is only a storage container.

Every market is converted to its own `dropna()` series before any lag, rolling window,
persistence rule, event label, or forward return is calculated.

Data before `2000-01-01` are retained only as warm-up history. Formal event counting,
labels, diagnostics, and research statistics begin on `CONFIG["research_start"]`.

In [ ]:
prices_wide = pd.read_parquet(SIGNAL_PATH).sort_index()
prices_wide = prices_wide.loc[
    prices_wide.index <= pd.Timestamp(CONFIG["research_end"])
].copy()

universe = pd.read_csv(UNIVERSE_PATH)

category_map = (
    universe.drop_duplicates("market")
    .set_index("market")["category"]
    .to_dict()
)

print("Wide panel shape:", prices_wide.shape)
print("Markets:", len(prices_wide.columns))
print("Date range:", prices_wide.index.min(), "to", prices_wide.index.max())

print(
    "Research window:",
    pd.Timestamp(CONFIG["research_start"]),
    "to",
    pd.Timestamp(CONFIG["research_end"]),
)


Wide panel shape: (10571, 53)
Markets: 53
Date range: 1990-01-01 00:00:00 to 2025-12-31 00:00:00
Research window: 2000-01-01 00:00:00 to 2025-12-31 00:00:00


## 3. Market-local conventional trend functions

In [ ]:
def tsmom_signal(price, lookback):
    p = price.dropna().astype(float).sort_index()
    trailing_return = p / p.shift(lookback) - 1.0
    return np.sign(trailing_return).fillna(0).astype(int)


def dual_ma_signal(price, fast_window, slow_window):
    p = price.dropna().astype(float).sort_index()

    fast_ma = p.rolling(
        fast_window,
        min_periods=fast_window
    ).mean()

    slow_ma = p.rolling(
        slow_window,
        min_periods=slow_window
    ).mean()

    return np.sign(fast_ma - slow_ma).fillna(0).astype(int)


def breakout_midpoint_signal(price, lookback):
    p = price.dropna().astype(float).sort_index()

    prior_high = (
        p.shift(1)
        .rolling(lookback, min_periods=lookback)
        .max()
    )

    prior_low = (
        p.shift(1)
        .rolling(lookback, min_periods=lookback)
        .min()
    )

    midpoint = (prior_high + prior_low) / 2.0
    return np.sign(p - midpoint).fillna(0).astype(int)


def ensemble_signal(a, b, c, min_votes=2):
    idx = a.index.intersection(b.index).intersection(c.index)

    a = a.reindex(idx)
    b = b.reindex(idx)
    c = c.reindex(idx)

    votes = a + b + c

    out = pd.Series(0, index=idx, dtype=int)
    out.loc[votes >= min_votes] = 1
    out.loc[votes <= -min_votes] = -1
    return out


def persistent_state(signal, required_days):
    s = signal.astype(int).copy()
    out = pd.Series(0, index=s.index, dtype=int)

    current = 0
    run_sign = 0
    run_length = 0

    for dt, value in s.items():
        value = int(value)

        if value == 0:
            run_sign = 0
            run_length = 0
            out.loc[dt] = current
            continue

        if value == run_sign:
            run_length += 1
        else:
            run_sign = value
            run_length = 1

        if run_length >= required_days:
            current = value

        out.loc[dt] = current

    return out

## 4. Event functions

A slow confirmation requires the new slow-ensemble sign to persist for the configured number of market-local trading days.

A fast counter-trend candidate occurs when:

- the slow benchmark remains in an established incumbent trend; and
- the faster conventional ensemble newly turns into the opposite direction.

This is intentionally simple. It is the control V2 must improve upon.

In [ ]:
def find_confirmed_flips(signal, persistence_days):
    s = signal.astype(int)
    events = []

    confirmed_state = 0
    i = 0
    n = len(s)

    while i < n:
        state = int(s.iloc[i])

        if state == 0 or state == confirmed_state:
            i += 1
            continue

        j = i
        while j < n and int(s.iloc[j]) == state:
            j += 1

        run_length = j - i

        if run_length >= persistence_days:
            if confirmed_state != 0 and state != confirmed_state:
                confirmation_pos = i + persistence_days - 1

                events.append({
                    "confirmation_date": s.index[confirmation_pos],
                    "from_state": int(confirmed_state),
                    "to_state": int(state),
                })

            confirmed_state = state

        i = j

    return pd.DataFrame(events)


def candidate_transition_dates(
    established_slow,
    fast_ensemble,
    cooldown_days
):
    idx = established_slow.index.intersection(fast_ensemble.index)

    slow = established_slow.reindex(idx).astype(int)
    fast = fast_ensemble.reindex(idx).astype(int)

    events = []
    last_event_pos = -10**9

    for i in range(1, len(idx)):
        incumbent = int(slow.iloc[i])
        current_fast = int(fast.iloc[i])
        prior_fast = int(fast.iloc[i - 1])

        if incumbent == 0:
            continue

        opposite = -incumbent

        newly_opposite = (
            current_fast == opposite
            and prior_fast != opposite
        )

        if not newly_opposite:
            continue

        if i - last_event_pos < cooldown_days:
            continue

        events.append({
            "candidate_date": idx[i],
            "incumbent_state": incumbent,
            "candidate_direction": opposite,
        })

        last_event_pos = i

    return pd.DataFrame(events)


def label_candidate_events(
    candidates,
    confirmations,
    horizon_days,
    market_index
):
    if candidates.empty:
        return pd.DataFrame()

    market_index = pd.Index(market_index).sort_values()
    rows = []

    for _, row in candidates.iterrows():
        candidate_date = pd.Timestamp(row["candidate_date"])
        candidate_direction = int(row["candidate_direction"])

        if candidate_date not in market_index:
            continue

        pos = market_index.get_loc(candidate_date)
        end_pos = min(pos + horizon_days, len(market_index) - 1)
        horizon_end = market_index[end_pos]

        eligible = confirmations.copy()

        if not eligible.empty:
            eligible["confirmation_date"] = pd.to_datetime(
                eligible["confirmation_date"]
            )

            eligible = eligible[
                (eligible["confirmation_date"] > candidate_date)
                & (eligible["confirmation_date"] <= horizon_end)
                & (eligible["to_state"] == candidate_direction)
            ].sort_values("confirmation_date")

        out = row.to_dict()
        out["label_horizon_end"] = horizon_end

        if eligible.empty:
            out["label"] = "failed"
            out["confirmation_date"] = pd.NaT
            out["lead_time_trading_days"] = np.nan
        else:
            conf_date = eligible.iloc[0]["confirmation_date"]
            conf_pos = market_index.get_loc(conf_date)

            out["label"] = "genuine"
            out["confirmation_date"] = conf_date
            out["lead_time_trading_days"] = conf_pos - pos

        rows.append(out)

    return pd.DataFrame(rows)


def forward_return(price, event_date, horizon):
    p = price.dropna().astype(float).sort_index()

    event_date = pd.Timestamp(event_date)

    if event_date not in p.index:
        return np.nan

    pos = p.index.get_loc(event_date)
    end_pos = pos + horizon

    if end_pos >= len(p):
        return np.nan

    return p.iloc[end_pos] / p.iloc[pos] - 1.0

## 5. Run the benchmark market by market

No trend calculation below uses the union-of-calendars panel directly.

In [ ]:
state_store = {}

confirmation_frames = []
candidate_frames = []

minimum_history = max(
    CONFIG["tsmom_slow_days"],
    CONFIG["ma_slow_slow_days"],
    CONFIG["breakout_slow_days"],
) + CONFIG["established_trend_days"]

traditional_research_start = pd.Timestamp(CONFIG["research_start"])
digital_assets_universe_start = pd.Timestamp(CONFIG["digital_assets_universe_start"])

def formal_research_start(market, price_series):
    """
    Determine the first date a market can contribute to formal research.

    Historical observations before the asset-class universe start may be
    used for indicator warm-up, but no formal event may occur before the
    universe-start date.

    Traditional assets:
        formal universe begins 2000-01-01

    Digital assets:
        formal universe begins 2017-01-01
    """

    category = category_map.get(market, "UNKNOWN")

    base_start = (
        digital_assets_universe_start
        if category == "DIGITAL_ASSETS"
        else traditional_research_start
    )

    p = price_series.dropna().sort_index()

    if len(p) < minimum_history:
        return None

    # Date on which the full required warm-up first becomes available.
    warmup_ready_date = p.index[minimum_history - 1]

    # Research cannot begin before the asset class enters the universe.
    target_start = max(base_start, warmup_ready_date)

    eligible_dates = p.index[p.index >= target_start]

    if len(eligible_dates) == 0:
        return None

    return eligible_dates[0]


for market in prices_wide.columns:

    p = prices_wide[market].dropna().sort_index().astype(float)

    if len(p) < minimum_history:
        print(f"Skipping {market}: insufficient local history ({len(p)} observations)")
        continue

    slow_tsmom = tsmom_signal(p, CONFIG["tsmom_slow_days"])
    slow_ma = dual_ma_signal(
        p,
        CONFIG["ma_fast_slow_days"],
        CONFIG["ma_slow_slow_days"],
    )
    slow_breakout = breakout_midpoint_signal(
        p,
        CONFIG["breakout_slow_days"],
    )

    fast_tsmom = tsmom_signal(p, CONFIG["tsmom_fast_days"])
    fast_ma = dual_ma_signal(
        p,
        CONFIG["ma_fast_fast_days"],
        CONFIG["ma_slow_fast_days"],
    )
    fast_breakout = breakout_midpoint_signal(
        p,
        CONFIG["breakout_fast_days"],
    )

    slow_ensemble = ensemble_signal(
        slow_tsmom,
        slow_ma,
        slow_breakout,
        CONFIG["ensemble_min_votes"],
    )

    fast_ensemble = ensemble_signal(
        fast_tsmom,
        fast_ma,
        fast_breakout,
        CONFIG["ensemble_min_votes"],
    )

    established_slow = persistent_state(
        slow_ensemble,
        CONFIG["established_trend_days"],
    )

    state_store[market] = {
        "price": p,
        "slow_tsmom": slow_tsmom,
        "slow_ma": slow_ma,
        "slow_breakout": slow_breakout,
        "fast_tsmom": fast_tsmom,
        "fast_ma": fast_ma,
        "fast_breakout": fast_breakout,
        "slow_ensemble": slow_ensemble,
        "fast_ensemble": fast_ensemble,
        "established_slow": established_slow,
    }

    market_research_start = formal_research_start(market, p)

    if market_research_start is None:
        print(
            f"Skipping {market}: insufficient post-entry history for "
            f"{minimum_history}-observation warm-up"
        )
        continue

    confirmations = find_confirmed_flips(
        slow_ensemble,
        CONFIG["confirmation_persistence_days"],
    )

    if not confirmations.empty:
        confirmations = confirmations[
            pd.to_datetime(confirmations["confirmation_date"]) >= market_research_start
        ].copy()

    if not confirmations.empty:
        confirmations["market"] = market
        confirmations["category"] = category_map.get(market, "UNKNOWN")
        confirmations["direction"] = np.where(
            confirmations["to_state"].eq(1),
            "bear_to_bull",
            "bull_to_bear",
        )
        confirmation_frames.append(confirmations)

    candidates = candidate_transition_dates(
        established_slow,
        fast_ensemble,
        CONFIG["candidate_cooldown_days"],
    )

    if not candidates.empty:
        candidates = candidates[
            pd.to_datetime(candidates["candidate_date"]) >= market_research_start
        ].copy()

    if not candidates.empty:
        candidates["market"] = market
        candidates["category"] = category_map.get(market, "UNKNOWN")
        candidates["direction"] = np.where(
            candidates["candidate_direction"].eq(1),
            "bear_to_bull",
            "bull_to_bear",
        )
        candidate_frames.append(candidates)

confirmation_events = (
    pd.concat(confirmation_frames, ignore_index=True)
    if confirmation_frames
    else pd.DataFrame()
)

candidate_events = (
    pd.concat(candidate_frames, ignore_index=True)
    if candidate_frames
    else pd.DataFrame()
)

print("Markets processed:", len(state_store))
print("Confirmation events:", len(confirmation_events))
print("Candidate events:", len(candidate_events))

Markets processed: 53
Confirmation events: 667
Candidate events: 3462


## 6. Label candidates as genuine or failed

Future information is used **only for the ex-post research label**.

No feature available after the candidate date is permitted to enter a later predictive model.

In [ ]:
labelled_frames = []

for market, states in state_store.items():

    market_candidates = (
        candidate_events[candidate_events["market"] == market].copy()
        if not candidate_events.empty
        else pd.DataFrame()
    )

    if market_candidates.empty:
        continue

    market_confirmations = (
        confirmation_events[confirmation_events["market"] == market].copy()
        if not confirmation_events.empty
        else pd.DataFrame()
    )

    labelled = label_candidate_events(
        market_candidates,
        market_confirmations,
        CONFIG["transition_horizon_days"],
        states["price"].index,
    )

    if labelled.empty:
        continue

    labelled["market"] = market
    labelled["category"] = category_map.get(market, "UNKNOWN")

    if "direction" not in labelled.columns:
        labelled["direction"] = np.where(
            labelled["candidate_direction"].eq(1),
            "bear_to_bull",
            "bull_to_bear",
        )

    labelled_frames.append(labelled)

labelled_events = (
    pd.concat(labelled_frames, ignore_index=True)
    if labelled_frames
    else pd.DataFrame()
)

if not labelled_events.empty:
    labelled_events = labelled_events.sort_values(
        ["category", "market", "candidate_date"]
    ).reset_index(drop=True)

print("Labelled events:", len(labelled_events))

if not labelled_events.empty:
    display(
        labelled_events.groupby(
            ["category", "direction", "label"]
        ).size().rename("events").reset_index()
    )

Labelled events: 3462


,category,direction,label,events
0,BONDS_RATES,bear_to_bull,failed,139
1,BONDS_RATES,bear_to_bull,genuine,63
2,BONDS_RATES,bull_to_bear,failed,152
3,BONDS_RATES,bull_to_bear,genuine,62
4,COMMODITIES,bear_to_bull,failed,325
5,COMMODITIES,bear_to_bull,genuine,193
6,COMMODITIES,bull_to_bear,failed,417
7,COMMODITIES,bull_to_bear,genuine,187
8,DIGITAL_ASSETS,bear_to_bull,failed,2
9,DIGITAL_ASSETS,bear_to_bull,genuine,5


## 7. Candidate-direction forward returns

In [ ]:
EVENT_RETURN_HORIZONS = [21, 63, 126]

event_returns = labelled_events.copy()

if not event_returns.empty:
    for horizon in EVENT_RETURN_HORIZONS:
        values = []

        for _, row in event_returns.iterrows():
            market = row["market"]
            raw_ret = forward_return(
                state_store[market]["price"],
                row["candidate_date"],
                horizon,
            )

            oriented_ret = (
                raw_ret * int(row["candidate_direction"])
                if pd.notna(raw_ret)
                else np.nan
            )

            values.append(oriented_ret)

        event_returns[f"oriented_return_{horizon}d"] = values

    display(
        event_returns.groupby(["category", "label"])[
            [f"oriented_return_{h}d" for h in EVENT_RETURN_HORIZONS]
        ].mean()
    )

oriented_return_21d  oriented_return_63d  \
category       label                                               
BONDS_RATES    failed             -0.001888            -0.009130   
               genuine             0.005838             0.015092   
COMMODITIES    failed             -0.017132            -0.040312   
               genuine             0.027477             0.079838   
DIGITAL_ASSETS failed              0.011622             0.016013   
               genuine             0.003883             0.088396   
FX             failed             -0.003757            -0.010234   
               genuine             0.010435             0.024408   
INDICES        failed             -0.024888            -0.047533   
               genuine             0.022115             0.062591   

                        oriented_return_126d  
category       label                          
BONDS_RATES    failed              -0.020671  
               genuine              0.026603  
COMMODITIES    failed              -0.065615  
               genuine              0.106164  
DIGITAL_ASSETS failed               0.022263  
               genuine              0.318355  
FX             failed              -0.011204  
               genuine              0.034977  
INDICES        failed              -0.066021  
               genuine              0.097436

## 8. Benchmark agreement diagnostics

Three-way agreement measures how often TSMOM, Dual MA and breakout all point in the same direction after all three are live.

In [ ]:
agreement_rows = []

for market, states in state_store.items():
    a = states["slow_tsmom"]
    b = states["slow_ma"]
    c = states["slow_breakout"]

    idx = a.index.intersection(b.index).intersection(c.index)
    a = a.reindex(idx)
    b = b.reindex(idx)
    c = c.reindex(idx)

    valid = a.ne(0) & b.ne(0) & c.ne(0)

    if valid.sum() == 0:
        continue

    agreement = (
        a.loc[valid].eq(b.loc[valid])
        & a.loc[valid].eq(c.loc[valid])
    )

    agreement_rows.append({
        "category": category_map.get(market, "UNKNOWN"),
        "market": market,
        "valid_observations": int(valid.sum()),
        "three_way_agreement_rate": float(agreement.mean()),
    })

benchmark_agreement = pd.DataFrame(agreement_rows)

display(
    benchmark_agreement.sort_values(
        ["category", "three_way_agreement_rate"],
        ascending=[True, False],
    )
)

,category,market,valid_observations,three_way_agreement_rate
1,BONDS_RATES,30-Day Fed Funds,6001,0.903849
3,BONDS_RATES,US 2Y Treasury Note,6082,0.824235
6,BONDS_RATES,US Ultra Treasury Bond,3718,0.802313
5,BONDS_RATES,US 5Y Treasury Note,6054,0.786422
4,BONDS_RATES,US 30Y Treasury Bond,6051,0.734094
0,BONDS_RATES,20+ Year Treasury ETF,5595,0.713315
2,BONDS_RATES,US 10Y Treasury Note,6033,0.710592
7,COMMODITIES,Aluminium,2597,0.846361
14,COMMODITIES,Gold,6054,0.809547
10,COMMODITIES,Coffee,6217,0.779797


## 9. Market and asset-class audit

This is a hard coverage check. If a market has valid price history but zero valid Dual-MA or breakout observations, stop and investigate before using the event results.

In [ ]:
audit_rows = []

for market, states in state_store.items():
    labels = (
        labelled_events[labelled_events["market"] == market]
        if not labelled_events.empty
        else pd.DataFrame()
    )

    confirmations_n = (
        int((confirmation_events["market"] == market).sum())
        if not confirmation_events.empty
        else 0
    )

    candidates_n = (
        int((candidate_events["market"] == market).sum())
        if not candidate_events.empty
        else 0
    )

    market_research_start = formal_research_start(market, states["price"])
    research_mask = states["price"].index >= market_research_start

    price_research = states["price"].loc[research_mask]
    slow_tsmom_research = states["slow_tsmom"].reindex(price_research.index)
    slow_ma_research = states["slow_ma"].reindex(price_research.index)
    slow_breakout_research = states["slow_breakout"].reindex(price_research.index)
    slow_ensemble_research = states["slow_ensemble"].reindex(price_research.index)
    established_research = states["established_slow"].reindex(price_research.index)

    audit_rows.append({
        "category": category_map.get(market, "UNKNOWN"),
        "market": market,
        "formal_research_start": market_research_start.date(),
        "actual_research_start": (
            price_research.index.min().date()
            if len(price_research)
            else None
        ),
        "research_end": (
            price_research.index.max().date()
            if len(price_research)
            else None
        ),
        "price_observations": len(price_research),
        "slow_tsmom_valid": int(slow_tsmom_research.ne(0).sum()),
        "slow_ma_valid": int(slow_ma_research.ne(0).sum()),
        "slow_breakout_valid": int(slow_breakout_research.ne(0).sum()),
        "slow_bull_days": int(slow_ensemble_research.eq(1).sum()),
        "slow_bear_days": int(slow_ensemble_research.eq(-1).sum()),
        "established_bull_days": int(established_research.eq(1).sum()),
        "established_bear_days": int(established_research.eq(-1).sum()),
        "confirmation_events": confirmations_n,
        "candidate_events": candidates_n,
        "genuine_events": (
            int(labels["label"].eq("genuine").sum())
            if not labels.empty
            else 0
        ),
        "failed_events": (
            int(labels["label"].eq("failed").sum())
            if not labels.empty
            else 0
        ),
    })

market_audit = pd.DataFrame(audit_rows)

summary_rows = []

for category, group in market_audit.groupby("category"):
    candidates_n = int(group["candidate_events"].sum())
    genuine_n = int(group["genuine_events"].sum())
    failed_n = int(group["failed_events"].sum())

    summary_rows.append({
        "asset_class": category,
        "markets": int(group["market"].nunique()),
        "confirmation_events": int(group["confirmation_events"].sum()),
        "candidates": candidates_n,
        "genuine": genuine_n,
        "failed": failed_n,
        "genuine_rate": (
            genuine_n / candidates_n
            if candidates_n > 0
            else np.nan
        ),
    })

asset_class_summary = pd.DataFrame(summary_rows).sort_values("asset_class")

def scope_row(scope_name, frame):
    candidates_n = int(frame["candidate_events"].sum())
    genuine_n = int(frame["genuine_events"].sum())
    failed_n = int(frame["failed_events"].sum())
    return {
        "scope": scope_name,
        "markets": int(frame["market"].nunique()),
        "confirmation_events": int(frame["confirmation_events"].sum()),
        "candidates": candidates_n,
        "genuine": genuine_n,
        "failed": failed_n,
        "genuine_rate": (
            genuine_n / candidates_n
            if candidates_n > 0
            else np.nan
        ),
    }

traditional_audit = market_audit[
    market_audit["category"] != "DIGITAL_ASSETS"
].copy()

digital_audit = market_audit[
    market_audit["category"] == "DIGITAL_ASSETS"
].copy()

scope_summary = pd.DataFrame([
    scope_row("ALL_ASSETS", market_audit),
    scope_row("TRADITIONAL_ASSETS_ONLY", traditional_audit),
    scope_row("DIGITAL_ASSETS_ONLY", digital_audit),
])

print("ASSET-CLASS SUMMARY")
display(asset_class_summary)

print("SCOPE SUMMARY")
display(scope_summary)

print("MARKET AUDIT")
display(
    market_audit.sort_values(
        ["category", "candidate_events", "market"],
        ascending=[True, False, True],
    )
)

bad_coverage = market_audit[
    (market_audit["price_observations"] >= minimum_history)
    & (
        (market_audit["slow_ma_valid"] == 0)
        | (market_audit["slow_breakout_valid"] == 0)
    )
]

if not bad_coverage.empty:
    raise RuntimeError(
        "Coverage gate failed: one or more markets have sufficient research-sample "
        "history but zero valid slow MA or breakout observations."
    )

print("Coverage gate: PASSED")

ASSET-CLASS SUMMARY


,asset_class,markets,confirmation_events,candidates,genuine,failed,genuine_rate
0,BONDS_RATES,7,83,416,125,291,0.300481
1,COMMODITIES,17,218,1122,380,742,0.338681
2,DIGITAL_ASSETS,1,6,31,14,17,0.451613
3,FX,10,159,717,260,457,0.362622
4,INDICES,18,201,1176,324,852,0.275510


SCOPE SUMMARY


,scope,markets,confirmation_events,candidates,genuine,failed,genuine_rate
0,ALL_ASSETS,53,667,3462,1103,2359,0.318602
1,TRADITIONAL_ASSETS_ONLY,52,661,3431,1089,2342,0.317400
2,DIGITAL_ASSETS_ONLY,1,6,31,14,17,0.451613


MARKET AUDIT


,category,market,formal_research_start,actual_research_start,research_end,price_observations,slow_tsmom_valid,slow_ma_valid,slow_breakout_valid,slow_bull_days,slow_bear_days,established_bull_days,established_bear_days,confirmation_events,candidate_events,genuine_events,failed_events
2,BONDS_RATES,US 10Y Treasury Note,2002-03-06,2002-03-06,2025-12-31,5986,5974,5986,5983,2089,2199,2696,3121,10,72,16,56
3,BONDS_RATES,US 2Y Treasury Note,2001-12-11,2001-12-11,2025-12-31,6052,6036,6052,6035,2609,2361,3631,2421,6,71,15,56
5,BONDS_RATES,US 5Y Treasury Note,2002-03-06,2002-03-06,2025-12-31,5998,5992,5998,5997,2277,2461,2729,3149,12,68,21,47
4,BONDS_RATES,US 30Y Treasury Bond,2002-03-05,2002-03-05,2025-12-31,5992,5989,5992,5991,2300,2142,3028,2729,15,67,20,47
0,BONDS_RATES,20+ Year Treasury ETF,2004-01-06,2004-01-06,2025-12-31,5533,5533,5533,5533,1867,2075,2693,2717,18,66,24,42
6,BONDS_RATES,US Ultra Treasury Bond,2011-06-17,2011-06-17,2025-12-31,3656,3656,3656,3655,1246,1720,1546,1997,12,38,14,24
1,BONDS_RATES,30-Day Fed Funds,2002-03-05,2002-03-05,2025-12-31,5982,5945,5982,5975,2444,2944,2913,3069,10,34,15,19
16,COMMODITIES,Natural Gas,2002-02-15,2002-02-15,2025-12-31,6002,5999,6002,6002,1911,2111,2785,3217,15,88,33,55
19,COMMODITIES,Silver,2002-02-14,2002-02-14,2025-12-31,5998,5998,5998,5998,2843,1748,4106,1798,11,78,21,57
9,COMMODITIES,Cocoa,2001-06-13,2001-06-13,2025-12-31,6158,6150,6158,6151,2388,1640,3948,2210,21,76,36,40


Coverage gate: PASSED


## 10. Save state panels and research outputs

The state panels are reconstructed onto the global panel index **only after** calculations have been completed market by market. They are saved for downstream notebooks, but must not be interpreted as implying a common trading calendar.

In [ ]:
global_index = prices_wide.index

def build_wide_state(key):
    out = pd.DataFrame(index=global_index)

    for market, states in state_store.items():
        out[market] = states[key].reindex(global_index)

    return out

slow_tsmom_wide = build_wide_state("slow_tsmom")
slow_ma_wide = build_wide_state("slow_ma")
slow_breakout_wide = build_wide_state("slow_breakout")
slow_ensemble_wide = build_wide_state("slow_ensemble")
fast_ensemble_wide = build_wide_state("fast_ensemble")
established_slow_wide = build_wide_state("established_slow")

slow_tsmom_wide.to_parquet(DATA_DIR / "slow_tsmom.parquet")
slow_ma_wide.to_parquet(DATA_DIR / "slow_dual_ma.parquet")
slow_breakout_wide.to_parquet(DATA_DIR / "slow_breakout.parquet")
slow_ensemble_wide.to_parquet(DATA_DIR / "slow_ensemble.parquet")
fast_ensemble_wide.to_parquet(DATA_DIR / "fast_ensemble.parquet")
established_slow_wide.to_parquet(DATA_DIR / "established_slow_trend.parquet")

confirmation_events.to_csv(
    RESULTS_DIR / "conventional_confirmation_events.csv",
    index=False,
)

candidate_events.to_csv(
    RESULTS_DIR / "fast_countertrend_candidates.csv",
    index=False,
)

labelled_events.to_csv(
    RESULTS_DIR / "labelled_transition_events.csv",
    index=False,
)

event_returns.to_csv(
    RESULTS_DIR / "transition_event_returns.csv",
    index=False,
)

benchmark_agreement.to_csv(
    RESULTS_DIR / "benchmark_agreement.csv",
    index=False,
)

asset_class_summary.to_csv(
    RESULTS_DIR / "v2_02_asset_class_summary.csv",
    index=False,
)

scope_summary.to_csv(
    RESULTS_DIR / "v2_02_scope_summary.csv",
    index=False,
)

market_audit.to_csv(
    RESULTS_DIR / "v2_02_market_audit.csv",
    index=False,
)

print("Saved V2.02 outputs to:", V202_ROOT)

Saved V2.02 outputs to: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.02


# Outputs to upload for analysis

## Required

1. `v2.02/results/v2_02_asset_class_summary.csv`
2. `v2.02/results/v2_02_market_audit.csv`
3. `v2.02/results/v2_02_scope_summary.csv`
4. `v2.02/results/labelled_transition_events.csv`
5. `v2.02/results/transition_event_returns.csv`
6. `v2.02/results/benchmark_agreement.csv`

## Preferred

7. `v2.02/results/conventional_confirmation_events.csv`
8. `v2.02/results/fast_countertrend_candidates.csv`

The scope summary explicitly compares the complete universe with traditional assets
excluding BTC and with `DIGITAL_ASSETS` alone.

In [ ]:
print("\n" + "=" * 80)
print("V2.02 REGENERATED BUILD COMPLETED SUCCESSFULLY")
print("=" * 80)

required_uploads = [
    RESULTS_DIR / "v2_02_asset_class_summary.csv",
    RESULTS_DIR / "v2_02_market_audit.csv",
    RESULTS_DIR / "v2_02_scope_summary.csv",
    RESULTS_DIR / "labelled_transition_events.csv",
    RESULTS_DIR / "transition_event_returns.csv",
    RESULTS_DIR / "benchmark_agreement.csv",
]

preferred_uploads = [
    RESULTS_DIR / "conventional_confirmation_events.csv",
    RESULTS_DIR / "fast_countertrend_candidates.csv",
]

print("\nREQUIRED OUTPUTS TO UPLOAD:")
for p in required_uploads:
    print(" -", p.name, "| exists:", p.exists())

print("\nPREFERRED OUTPUTS:")
for p in preferred_uploads:
    print(" -", p.name, "| exists:", p.exists())

missing = [p for p in required_uploads if not p.exists()]
if missing:
    raise RuntimeError(
        "V2.02 completion check failed. Missing required outputs: "
        + ", ".join(p.name for p in missing)
    )

print("\nCompletion check: PASSED")



V2.02 REGENERATED BUILD COMPLETED SUCCESSFULLY

REQUIRED OUTPUTS TO UPLOAD:
 - v2_02_asset_class_summary.csv | exists: True
 - v2_02_market_audit.csv | exists: True
 - v2_02_scope_summary.csv | exists: True
 - labelled_transition_events.csv | exists: True
 - transition_event_returns.csv | exists: True
 - benchmark_agreement.csv | exists: True

PREFERRED OUTPUTS:
 - conventional_confirmation_events.csv | exists: True
 - fast_countertrend_candidates.csv | exists: True

Completion check: PASSED


## Research Outcome

Book 02 established the conventional trend-following benchmark and transformed the qualitative concept of a "trend transition" into a reproducible event-classification problem.

Slow conventional trend state is defined using an ensemble of:

- 12-month time-series momentum;
- slow dual moving averages;
- slow breakout/channel information.

Faster versions of the same conventional trend families are used as transition-candidate generators and falsification controls rather than being treated as transition alpha themselves.

A candidate transition is subsequently classified ex post as either:

- **Genuine** — the incumbent trend ultimately gives way to a conventionally confirmed opposite trend; or
- **Failed** — the apparent transition resolves without durable opposite-trend confirmation.

All rolling calculations, persistence rules, lags, forward horizons and event labels are computed on each market's own local trading calendar before markets are recombined. This prevents missing observations in one market from changing the effective horizons of another.

Across the 53-market universe, Book 02 identifies:

- **3,462 candidate transitions**;
- **1,103 genuine transitions**;
- **2,359 failed transitions**;
- an unconditional genuine-transition rate of **31.86%**.

Excluding Bitcoin produces 3,431 traditional-asset candidates with a genuine rate of **31.74%**, demonstrating that the transition-event population is not materially driven by the digital-asset sleeve.

The approximately 32% base rate establishes a demanding but economically meaningful prediction problem. Most apparent early reversals fail, which is precisely the structural difficulty an anticipatory transition strategy must overcome.

### Conclusion

Book 02 establishes the central empirical problem for V2:

\[
P(\text{Genuine Transition} \mid \text{information available at candidate date})
\]

rather than attempting to predict exact market tops or bottoms.

The conventional trend benchmark therefore serves two roles: it defines the incumbent market state and provides the external confirmation event against which anticipatory information can be evaluated.

**Status: FROZEN as the conventional-trend and transition-event benchmark.**